# Phân tích Sales & Customer — Olist

Release `87f01740-0523-5fd4-be0e-eb52acb2842a` · source cut 1 · tháng phân tích **2018-07** · cutoff khách hàng **31/08/2018**.

## 1. Bán hàng đang ở mức nào?

Tháng được chọn có **867,953.46 BRL** giá trị hàng thuộc đơn delivered, **6,159** đơn và **6,100** khách hàng phân biệt. AOV **140.92 BRL/đơn**; cancellation rate **0.65%**.

Giá trị hàng thay đổi **1.39%** so với đúng tháng trước và **80.22%** so với cùng tháng năm trước. Đây là mô tả snapshot, chưa kiểm soát độ đầy đủ của nguồn hoặc thay đổi cơ cấu; không suy ra quan hệ nhân quả. Kỳ cuối 2018-08 cần thận trọng về right-censoring nên không được dùng mặc định để kết luận doanh nghiệp suy giảm.

## 2. Nên chú ý danh mục/người bán nào?

Category đứng đầu là **health_beauty**, chiếm **11.93%** giá trị delivered của tháng. Top 10 sellers chiếm **14.15%**. Có thể dùng danh sách này để chọn đối tượng kiểm tra vận hành/nguồn cung trước; dataset không có cost, margin hoặc inventory để khuyến nghị tăng lợi nhuận hay bổ sung tồn kho.

| category | delivered_merchandise_value | delivered_item_rows |
| --- | --- | --- |
| health_beauty | 103,524.22 | 770 |
| watches_gifts | 95,164.79 | 521 |
| housewares | 62,583.30 | 564 |
| bed_bath_table | 54,405.23 | 617 |
| sports_leisure | 54,015.89 | 460 |

## 3. Khách hàng có mua lại không?

Có **2,801/93,358** khách từng có ít nhất hai đơn delivered, tương đương **3.00%**. Nhóm này đóng góp **5.51%** merchandise value quan sát. Tỷ lệ này bị giới hạn bởi cửa sổ dữ liệu và định danh nguồn; không gọi nhóm một đơn là churn, không suy ra customer lifetime value tương lai.

Cohort dựa trên lần mua delivered đầu được quan sát; khách được đếm distinct lại trong từng tháng tuổi cohort. Mỗi cell chia cho cohort size gốc; chỉ tạo cell nằm trong cửa sổ đã quan sát. Các ô không có hoạt động trong cửa sổ là 0, ô tương lai không được tạo.

| frequency_segment | customers | delivered_orders | observed_merchandise_value |
| --- | --- | --- | --- |
| 1 delivered order | 90,557 | 90,557 | 12,493,089.36 |
| 2 delivered orders | 2,573 | 5,146 | 631,274.90 |
| 3+ delivered orders | 228 | 775 | 97,133.85 |

## 4. Giao hàng và thanh toán có gì đáng chú ý?

Trong tháng có **6,156** đơn đủ điều kiện ngày giao; **5,948** đơn đúng hẹn (**96.62%**). Ngày hứa giao được so theo ngày lịch, đúng quy tắc fact. Các bang có ít quan sát cần được xem cùng sample size; không xếp hạng chỉ từ trung bình.

| buyer_state | eligible_deliveries | on_time_rate | median_delivery_days |
| --- | --- | --- | --- |
| SP | 2,712 | 96.31% | 5.79 |
| RJ | 696 | 97.56% | 8.06 |
| MG | 644 | 98.76% | 7.15 |
| PR | 314 | 98.09% | 7.37 |
| RS | 308 | 98.38% | 9.31 |
| BA | 245 | 93.88% | 12.14 |
| SC | 195 | 97.95% | 8.29 |
| DF | 165 | 95.76% | 7.91 |

| payment_method | payment_records | orders_using_method | recorded_payment_value |
| --- | --- | --- | --- |
| credit_card | 4,755 | 4,738 | 803,674.49 |
| boleto | 1,229 | 1,229 | 198,041.24 |
| debit_card | 242 | 242 | 44,866.18 |
| voucher | 281 | 212 | 19,958.84 |

Payment records không phải số đơn: một đơn có thể có nhiều records/phương thức. Recorded amount có thể gồm freight hoặc thuộc trạng thái khác; không gọi đây là doanh thu thuần, tiền đã thu hay refunds.

## 5. Chất lượng dữ liệu có làm sai kết luận không?

| rule_id | severity | affected_records |
| --- | --- | --- |
| DQ-CAT-001 | warning | 610 |
| DQ-COV-001 | warning | 775 |
| DQ-COV-002 | warning | 1 |
| DQ-PAY-001 | warning | 303 |
| DQ-TIME-001 | warning | 8 |
| DQ-TIME-002 | warning | 166 |

Những cảnh báo này được giữ nguyên từ nguồn. Lỗi schema/grain/relationship/parity mới chặn publication. Thiếu items/payment không bị tự điền tiền để cân số. Không cộng affected occurrences giữa các rule thành số bản ghi lỗi phân biệt.

## 6. Tái lập và giới hạn

`make analyze` chạy SQL trong `analysis/sql/`, EXPLAIN trước execute, ghi CSV, query SHA và kiểm oracle vào `var/analysis/results.json`. Notebook đi kèm đọc chính các CSV đã xuất. Các core KPI đối soát với MetricFlow; cohort/recency/payment là governed SQL analysis và không được trình bày như metric MetricFlow mới.

Nguồn: Olist v2, CC BY-NC-SA 4.0; portfolio độc lập. BRL là quy ước vì CSV không có currency. Không có tên khách/sản phẩm, chi phí, clinical/telesales hoặc lịch sử trạng thái tại ngày mua. Phân tích mô tả các quan sát trong snapshot, không đại diện tình hình hiện tại của Abbott/Concentrix.


In [1]:
from pathlib import Path
import csv, json
base = Path('.')
results = json.loads((base / 'results.json').read_text())
assert results['passed']
print(results['release'])
print(results['checks'])


{'release_id': '87f01740-0523-5fd4-be0e-eb52acb2842a', 'pipeline_run_id': '5ac008ff-3041-54cf-b03a-b7e76a5fa82f', 'source_cut': 1}
{'eight_core_metrics_match_order_grain': True, 'customer_counts_and_value_reconcile': True, 'cohort_oracle_and_denominators': True, 'payment_grain_count': True}


### categories
SQL: `analysis/sql/categories.sql`; số dòng: 67.

In [2]:
with (base / 'categories.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 67


[{'category': 'health_beauty',
  'delivered_merchandise_value': '103524.22',
  'delivered_item_rows': '770',
  'value_share': '0.11927392973351359185',
  'cumulative_value_share': '0.11927392973351359185'},
 {'category': 'watches_gifts',
  'delivered_merchandise_value': '95164.79',
  'delivered_item_rows': '521',
  'value_share': '0.10964273360924213609',
  'cumulative_value_share': '0.22891666334275572794'},
 {'category': 'housewares',
  'delivered_merchandise_value': '62583.30',
  'delivered_item_rows': '564',
  'value_share': '0.07210444209762122499',
  'cumulative_value_share': '0.30102110544037695293'},
 {'category': 'bed_bath_table',
  'delivered_merchandise_value': '54405.23',
  'delivered_item_rows': '617',
  'value_share': '0.06268219726896416773',
  'cumulative_value_share': '0.36370330270934112066'},
 {'category': 'sports_leisure',
  'delivered_merchandise_value': '54015.89',
  'delivered_item_rows': '460',
  'value_share': '0.06223362483052950788',
  'cumulative_value_share

### cohorts
SQL: `analysis/sql/cohorts.sql`; số dòng: 140.

In [3]:
with (base / 'cohorts.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 140


[{'cohort_month': '2016-09-01',
  'month_offset': '0',
  'cohort_size': '1',
  'active_customers': '1',
  'observed_retention': '1.00000000000000000000'},
 {'cohort_month': '2016-09-01',
  'month_offset': '1',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month': '2016-09-01',
  'month_offset': '2',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month': '2016-09-01',
  'month_offset': '3',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month': '2016-09-01',
  'month_offset': '4',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month': '2016-09-01',
  'month_offset': '5',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month': '2016-09-01',
  'month_offset': '6',
  'cohort_size': '1',
  'active_customers': '0',
  'observed_retention': '0E-20'},
 {'cohort_month':

### customer_segments
SQL: `analysis/sql/customer_segments.sql`; số dòng: 3.

In [4]:
with (base / 'customer_segments.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 3


[{'frequency_segment': '1 delivered order',
  'customers': '90557',
  'delivered_orders': '90557',
  'observed_merchandise_value': '12493089.36',
  'average_recency_days': '240.0254094106474375'},
 {'frequency_segment': '2 delivered orders',
  'customers': '2573',
  'delivered_orders': '5146',
  'observed_merchandise_value': '631274.90',
  'average_recency_days': '223.4057520404197435'},
 {'frequency_segment': '3+ delivered orders',
  'customers': '228',
  'delivered_orders': '775',
  'observed_merchandise_value': '97133.85',
  'average_recency_days': '203.7938596491228070'}]

### delivery
SQL: `analysis/sql/delivery.sql`; số dòng: 27.

In [5]:
with (base / 'delivery.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 27


[{'buyer_state': 'SP',
  'orders': '2777',
  'eligible_deliveries': '2712',
  'on_time_deliveries': '2612',
  'on_time_rate': '0.96312684365781710914',
  'median_delivery_days': '5.787100694444444',
  'average_delivery_days': '6.7126497545203758',
  'canceled_orders': '22'},
 {'buyer_state': 'RJ',
  'orders': '717',
  'eligible_deliveries': '696',
  'on_time_deliveries': '679',
  'on_time_rate': '0.97557471264367816092',
  'median_delivery_days': '8.062951388888889',
  'average_delivery_days': '9.5644682245104300',
  'canceled_orders': '2'},
 {'buyer_state': 'MG',
  'orders': '658',
  'eligible_deliveries': '644',
  'on_time_deliveries': '636',
  'on_time_rate': '0.98757763975155279503',
  'median_delivery_days': '7.148703703703704',
  'average_delivery_days': '8.4247385768921095',
  'canceled_orders': '7'},
 {'buyer_state': 'PR',
  'orders': '320',
  'eligible_deliveries': '314',
  'on_time_deliveries': '308',
  'on_time_rate': '0.98089171974522292994',
  'median_delivery_days': '7.37

### monthly
SQL: `analysis/sql/monthly.sql`; số dòng: 25.

In [6]:
with (base / 'monthly.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 25


[{'release_id': '87f01740-0523-5fd4-be0e-eb52acb2842a',
  'month': '2016-09-01',
  'total_orders': '4',
  'delivered_orders': '1',
  'canceled_orders': '2',
  'cancellation_rate': '0.5',
  'delivered_merchandise_value': '134.97',
  'delivered_aov': '134.97',
  'delivered_unique_customers': '1',
  'delivered_item_rows': '3',
  'generated_sql_sha256': '5f3c9764142999cde2ed81c7dd40f8ec660f1c291447f410aba053a2a965783e',
  'previous_month_value': '',
  'value_mom': '',
  'previous_year_value': '',
  'value_yoy': ''},
 {'release_id': '87f01740-0523-5fd4-be0e-eb52acb2842a',
  'month': '2016-10-01',
  'total_orders': '324',
  'delivered_orders': '265',
  'canceled_orders': '24',
  'cancellation_rate': '0.07407407407407407',
  'delivered_merchandise_value': '40325.11',
  'delivered_aov': '152.17022641509433',
  'delivered_unique_customers': '262',
  'delivered_item_rows': '313',
  'generated_sql_sha256': '5f3c9764142999cde2ed81c7dd40f8ec660f1c291447f410aba053a2a965783e',
  'previous_month_value

### payments
SQL: `analysis/sql/payments.sql`; số dòng: 4.

In [7]:
with (base / 'payments.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 4


[{'payment_method': 'credit_card',
  'payment_records': '4755',
  'orders_using_method': '4738',
  'recorded_payment_value': '803674.49',
  'average_installments': '3.3396424815983176'},
 {'payment_method': 'boleto',
  'payment_records': '1229',
  'orders_using_method': '1229',
  'recorded_payment_value': '198041.24',
  'average_installments': '1.00000000000000000000'},
 {'payment_method': 'debit_card',
  'payment_records': '242',
  'orders_using_method': '242',
  'recorded_payment_value': '44866.18',
  'average_installments': '1.00000000000000000000'},
 {'payment_method': 'voucher',
  'payment_records': '281',
  'orders_using_method': '212',
  'recorded_payment_value': '19958.84',
  'average_installments': '1.00000000000000000000'}]

### quality
SQL: `analysis/sql/quality.sql`; số dòng: 6.

In [8]:
with (base / 'quality.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 6


[{'rule_id': 'DQ-CAT-001', 'severity': 'warning', 'affected_records': '610'},
 {'rule_id': 'DQ-COV-001', 'severity': 'warning', 'affected_records': '775'},
 {'rule_id': 'DQ-COV-002', 'severity': 'warning', 'affected_records': '1'},
 {'rule_id': 'DQ-PAY-001', 'severity': 'warning', 'affected_records': '303'},
 {'rule_id': 'DQ-TIME-001', 'severity': 'warning', 'affected_records': '8'},
 {'rule_id': 'DQ-TIME-002', 'severity': 'warning', 'affected_records': '166'}]

### sellers
SQL: `analysis/sql/sellers.sql`; số dòng: 1261.

In [9]:
with (base / 'sellers.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('Rows:', len(rows))
rows[:10]


Rows: 1261


[{'seller_id': '1025f0e2d44d7041d6cf58b6550e0bfa',
  'delivered_merchandise_value': '17465.26',
  'delivered_item_rows': '151',
  'value_share': '0.02012234619123472358',
  'cumulative_value_share': '0.02012234619123472358'},
 {'seller_id': '04308b1ee57b6625f47df1d56f00eedf',
  'delivered_merchandise_value': '16259.30',
  'delivered_item_rows': '27',
  'value_share': '0.01873291685478159163',
  'cumulative_value_share': '0.03885526304601631521'},
 {'seller_id': '612170e34b97004b3ba37eae81836b4c',
  'delivered_merchandise_value': '15770.33',
  'delivered_item_rows': '72',
  'value_share': '0.01816955715574888082',
  'cumulative_value_share': '0.05702482020176519603'},
 {'seller_id': '6560211a19b47992c3666cc44a7e94c0',
  'delivered_merchandise_value': '13237.00',
  'delivered_item_rows': '207',
  'value_share': '0.01525081771089431454',
  'cumulative_value_share': '0.07227563791265951057'},
 {'seller_id': '2eb70248d66e0e3ef83659f71b244378',
  'delivered_merchandise_value': '13036.28',
  